In [ ]:
# Install required libraries
!pip install ultralytics albumentations opencv-python-headless

import os
import cv2
import glob
import albumentations as A
from ultralytics import YOLO

In [ ]:
import albumentations as A
import cv2

def get_edge_augmentation(img_size=640):
    return A.Compose([
        
        # 1. Simulate low-resolution sensor
        A.OneOf([
            A.Downscale(scale_min=0.2, scale_max=0.5, interpolation=cv2.INTER_LINEAR),
        ], p=0.7),

        # 2. Resize back to training size (Crucial for YOLO architecture)
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_CONSTANT, value=0), # value=0 means black padding

        # 3. JPEG compression artifacts
        A.ImageCompression(quality_lower=30, quality_upper=70, p=0.7),

        # 4. Sensor noise
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0)),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5)),
        ], p=0.6),

        # 5. Blur (motion + defocus)
        A.OneOf([
            A.MotionBlur(blur_limit=7),
            A.GaussianBlur(blur_limit=5),
            A.MedianBlur(blur_limit=5),
        ], p=0.5),

        # 6. Lighting degradation
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
            A.RandomGamma(gamma_limit=(50, 150)),
        ], p=0.6),

        # 7. Optional: slight color distortion (cheap sensors)
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),

    ], bbox_params=A.BboxParams(
        format='yolo', 
        label_fields=['class_labels'],
        min_visibility=0.2,   # Drop boxes if 80% of it is cut off by padding/resizing
        min_area=10.0         # Drop boxes if they become smaller than 10 pixels in area
    ))

# Implementation:
aug = get_edge_augmentation()

# Ensure your 'bboxes' are already converted from COCO to YOLO format before passing them here!
try:
    transformed = aug(image=image, bboxes=bboxes, class_labels=labels)
    transformed_image = transformed["image"]
    transformed_bboxes = transformed["bboxes"]
    transformed_labels = transformed["class_labels"]
except ValueError as e:
    print(f"Skipping image due to augmentation error: {e}")

In [ ]:
# Phase 1: Training the Baseline Model
from ultralytics import YOLO

# Load the pretrained baseline model (YOLOv8s for this example)
model = YOLO("yolov8s.pt")

print("Starting Fine-tuning Phase...")
results = model.train(
    data="coco_iot.yaml", # Your config pointing to the new noisy dataset folders
    epochs=100,           # Standard benchmark length
    imgsz=416,            # Edge device resolution
    batch=16,             # Adjust based on your GPU vRAM
    freeze=10,            # FREEZE THE BACKBONE (Layers 0-10) as per paper methodology
    device=0,             # GPU id (e.g. 0, 1 for your HPC)
    optimizer="auto",     # AdamW is usually selected
    project="edge_benchmarks",
    name="yolo8s_iot_finetune"
)

In [ ]:
print("Evaluating Baseline Model (FP32)...")
# Load the best weights from the fine-tuning phase
best_model = YOLO("edge_benchmarks/yolo8s_iot_finetune/weights/best.pt")

# Validate the model on the Noisy Validation set
base_metrics = best_model.val(data="coco_iot.yaml", split='val')

print(f"Baseline mAP50: {base_metrics.box.map50}")
print(f"Baseline mAP50-95: {base_metrics.box.map}")

In [ ]:
print("Starting INT8 Quantization Process...")

# For NVIDIA Edge Devices (Jetson Nano, Orin, etc.), we export to TensorRT engine
quantized_engine = best_model.export(
    format="engine",       # Export to TensorRT (.engine)
    int8=True,             # Trigger INT8 Quantization
    dynamic=True,          # Allows dynamic batch sizes during inference
    workspace=4,           # GiB of memory allocation for conversion
    data="coco_iot.yaml",  # Dataset required for INT8 calibration (crucial!)
    fraction=0.1           # Use 10% of dataset to calibrate the INT8 scales
)

# NOTE: If deploying to a Raspberry Pi or TPU, change format="engine" to format="tflite" or format="edgetpu"

In [ ]:
print("Evaluating Quantized Model (INT8)...")

# Load the quantized model
# The exported file will be named "best.engine" (or "best.tflite")
int8_model = YOLO("edge_benchmarks/yolo8s_iot_finetune/weights/best.engine", task="detect")

# Validate on the Noisy Validation set
quantized_metrics = int8_model.val(data="coco_iot.yaml", split='val')

print("\n--- FINAL BENCHMARK COMPARISON ---")
print(f"FP32 Baseline mAP50: {base_metrics.box.map50}")
print(f"INT8 Quantized mAP50: {quantized_metrics.box.map50}")

# You can also use model.predict() on a single image and use the `time` module to measure latency (ms)